In [1]:
## compute Tw from T and relative humidity using Stull (2011) empirical approximation
#### 6/30/26

In [2]:
# imports
import os
import xarray as xr
import numpy as np
import netCDF4 
import glob
import pandas as pd
import geopandas as gpd
from datetime import datetime
from scipy import stats

In [3]:
# interactive plotting stuff 
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colors
import matplotlib.cm as cm
import matplotlib.gridspec as gridspec
from matplotlib import rcParams
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.lines import Line2D 

#import matplotlib.dates as mdates
%matplotlib inline
plt.rcParams['figure.figsize'] = 12, 6
#%config InlineBackend.figure_format = 'retina'

import cartopy
import cartopy.crs as ccrs
from cartopy.util import add_cyclic_point

In [4]:
def reshape_1d_to_2d_331(da):
    """
    This function takes a 1-d timeseries of daily data (dims = [day]) to 2-d (dims = [year, day of year (1-366)])
    which makes it possible to remove the linear trend from each day's data. 
    
    Args:
        da (xr.DataArray) (must be 1-d, have a dim labeled "time", must be daily data)
        
    Returns
        yr_day_2d_da (xr.DataArray)
        
    3/31/24
    """

    # calculate number of years in the data, save important initial indices
    numyrs = len(np.unique(da.time.dt.year))
    yrlist = np.unique(da.time.dt.year)
    firstyr_num = yrlist[0]
    firstday_num = da.time.dt.dayofyear[0].item()

    # allocate blank array that has dims [row=year, columns=day of year (1-366)]
    yr_day_2darr = np.zeros(shape=(numyrs, 366)) * np.nan

    # counter var's
    firstday_idx = firstday_num - 1
    yr = firstyr_num
    yr_idx = 0
    dayctr = 0 # dayctr tracks the index of the observation within the raw 1-d data that we want to select

    dat = da.data

    while yr_idx < numyrs:

        # check if leap year:
        if ((yr - 1900) % 4) == 0:
                n_daysinyear = 366
        else: 
                n_daysinyear = 365


#        print(yr_idx, yr, n_daysinyear)

        if yr_idx == 0:

            day_of_year_idx = firstday_idx

            # in year 0: determine how many days worth of data to select (depends on when dataset starts)
            n_days2select = (n_daysinyear - firstday_idx)

        if yr_idx > 0: 

                day_of_year_idx = 0

                # assume that every year besides first and last has a full 365 or 366 entries
                # (even if some are NaN)
                n_days2select = n_daysinyear

                # if we are in the last year of data:
                if yr_idx == (numyrs-1):

                    # only want to select the days exist in data, in other words, this guards against
                    # the case that the final year of data lacks a full year of entries (ends before 12/31). 
                    n_days2select = (len(dat) - dayctr)


#        print(yr_idx, yr, n_daysinyear, firstday_idx, n_days2select)

        # select data
        year_sel = dat[dayctr:(dayctr+n_days2select)]

        # put it into the array
        yr_day_2darr[yr_idx, (day_of_year_idx):(day_of_year_idx+n_days2select)] = year_sel

        # advance day index, year index 
        dayctr = (dayctr + n_days2select)
        yr+=1
        yr_idx+=1

    
    # put into xr.dataArray
    yr_day_2d_da = xr.DataArray(yr_day_2darr, dims=['year', 'day'], 
                                coords={'year':yrlist, 'day':np.array(range(1, 367))})
    
    return yr_day_2d_da

def detrend_dim_smoothedv2(da, da_roll, dim, deg=1):
    """
    From: https://gist.github.com/rabernat/1ea82bb067c3273a6166d1b1f77d490f
    
    UPDATED 11/20/25 to allow for the passing of 2 arrays: da and da_roll, 
    where da_roll is a smoothed version of da, and we use da_roll to compute the 
    linear trends in each day. We then use these (possibly-less-noisy/unreliable?) trends
    which were computed using the smoother data to detrend the observed "da."
    """
    
    # detrend along a single dimension
    p = da_roll.polyfit(dim=dim, deg=deg) # note using da_roll here
    fit = xr.polyval(da_roll[dim], p.polyfit_coefficients)
    
    return da - fit # subtracting the fit from the NORMAL (unsmoothed) data

def anomalize_dailydata_via_lineartrend_roll11(da):
    """
    this function combines the above two (reshape_1d_to_2d_331) and (detrend_dim)
    plus has to do some pesky reshaping
    
    - with update 11/20/25, generates and passes 11-day rolling data to use for the detrending
    
    3/31/24 - Last updated 11/20/25
    """
    
    # save some original values
    originaldates = da.time
    firstday_num = da.time.dt.dayofyear[0].item()
    firstday_idx = firstday_num - 1
    numdates = len(da)

    ## turn the data from 1-d to 2-d for ease of detrending ##
    da_2d = reshape_1d_to_2d_331(da)
    
    ## also generate 11-day rolling data, reshape that to 2d (11/20/25)
    da_2d_r11 = reshape_1d_to_2d_331(da.rolling(time=11, center=True, min_periods=9).mean()) # min periods decision sorta backed up by data...
    
    ## detrend ##
    da_2d_detrended = detrend_dim_smoothedv2(da=da_2d, da_roll=da_2d_r11, dim='year', deg=1)
    
    # now I have 2-d data that is detrended, however, there are more entries than in 
    # the original input "da" because I've added a bunch of blank spaces for day no. 366 in all the 
    # non-leap years. 

    # this mask identifies dates that have nodata & day num = 366
    empty_leapmask = (((da_2d_detrended.year-1900)%4)>0) * (da_2d_detrended.day==366)
    
    # flatten the 2-d detrended data and the mask
    datflat = da_2d_detrended.data.reshape(-1)
    empty_leapmask_flat = empty_leapmask.data.reshape(-1)

    # mask out the NaN's that came from the leap year column
    datflat_drop_leapNaNs = datflat[~empty_leapmask_flat]

    # cut out any leading or trailing NaN's that could've happened due to incomplete data 
    final_detrened_datflat = datflat_drop_leapNaNs[firstday_idx:(firstday_idx+numdates)]

    # put into dataArray
    detrended_flat_da = xr.DataArray(final_detrened_datflat, dims='time', coords={'time':originaldates})
    
    return detrended_flat_da

def gen_11d_smoothed_stds(da):
    '''
    11/20/25
    '''

    # first, gen standard deviations for every day of year (individually)
    stds = da.groupby('time.dayofyear').std()

    ## apply 11-day rolling window to smooth daily S-D's

    # i need to extend the 'dayofyear' dim in order to get the rolling val's for the 11days near the start b/c otherwise they will be NaN
    ext_arr = np.zeros(shape=400)
    ext_arr[:366] = stds
    ext_arr[366:] = stds[:(400-366)] # beyond day 366, fill in with the beginning of the year again

    # create the 11-day rolling using the extended array
    ext_arr_da = xr.DataArray(ext_arr, dims='dayofyear', coords={'dayofyear':np.array(range(400))+1})
    ext_arr_da11roll = ext_arr_da.rolling(dayofyear=11, center=True).mean()

    # fill this back in on a 366 day-long array
    rolling_arr = np.zeros(shape=366)
    rolling_arr[:] = ext_arr_da11roll[:366]
    rolling_arr[:11] = ext_arr_da11roll[366:377]

    # put back into an xr.DataArray
    stds_11roll_da = xr.DataArray(rolling_arr, dims='dayofyear', coords={'dayofyear':stds.dayofyear})

    return stds_11roll_da

In [5]:
script = os.getcwd() + '/prep_detredanom_daily_Tw_est.ipynb'

# compute (estimated) Tw

In [6]:
# open relative humidity and t(mean) data
rh_da = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.rh.nc').rh

# tmean
tx_da = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.tx.nc').Tx
tn_da = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.tn.nc').Tn

tmean_da = (tx_da + tn_da) / 2

In [7]:
# generate mask for which this equation can be used / is valid:
rh_mask = (rh_da >= 5) * (rh_da <= 99)
t_mask = (tmean_da >= -20) * (tmean_da <= 50)
mask = (rh_mask * t_mask)

In [8]:
# compute estimated Tw from the Stull 2011 (https://doi.org/10.1175/JAMC-D-11-0143.1, equation 1)
Tw_est_da = (tmean_da * np.arctan(0.15177 * ((rh_da + 8.313659)**(1/2)))) + np.arctan(tmean_da + rh_da) - np.arctan(rh_da - 1.676331) + (0.00391839 * (rh_da**(3/2)) * np.arctan(0.023101 * rh_da)) - 4.686035

In [9]:
# mask out for invalid rh or T values, send to dataset
ds_out = Tw_est_da.where(mask).to_dataset(name='Tw')

ds_out.attrs['script'] = script
now = datetime.now()
ds_out.attrs['timestamp'] = now.strftime("%Y-%m-%d %H:%M:%S")
ds_out.attrs['desc.'] = 'Tw estimated using Tmean and RH. From Stull, 2011 eq. 1 (https://doi.org/10.1175/JAMC-D-11-0143.1)'
ds_out.to_netcdf('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.Tw.nc')

In [10]:
# I've sanity-checked it online compared to some Tw calculators and it looks OK? I'm assuming they use the same simplified equation. 

# compute anomalies via linear detrend w/11-day rolling window

In [11]:
clim_pd = ['1990-01-01', '2023-12-31']
period = clim_pd # (for now)

# dataframe with the stations we are using
df = pd.read_csv('/home/nsiegert/projects/coastal_sst/data/hadisd_stations_using_Expanded.csv')
df = df.drop(['Unnamed: 0'], axis=1)
stalist = df.STAID
df

# open the heatwave ds
hw_ds = xr.open_dataset('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.1.5deg.marineheatwaves.nc')
hw_ds_time0 = hw_ds.time[0]
hw_ds_time1 = hw_ds.time[-1]
hw_ds_n_times = len(hw_ds.time)

In [12]:
## Prep data for each station ##

print('prepping data for each station. 1.', flush=True)

# output arrays
var_arr = np.zeros(shape=hw_ds.MHW.shape) * np.NaN
var_det_arr = np.zeros(shape=hw_ds.MHW.shape) * np.NaN
var_STdet_arr = np.zeros(shape=hw_ds.MHW.shape) * np.NaN

messed_up_data_staids = []

prepping data for each station. 1.


In [13]:
%%time 

# for each station,
for i, staid in enumerate(stalist):
#    print(i, staid)

    # sel daily Tw
    da = ds_out.Tw[i, :]

    # perform linear anomaly detrending (new version 11/20/25)
    detrend_anom_da = anomalize_dailydata_via_lineartrend_roll11(da)
    
    ## Compute Std. Anomalies ## (new section 11/20/25)
    detrend_stanom = (detrend_anom_da.groupby('time.dayofyear') / gen_11d_smoothed_stds(da=da))

    # put into arrays.
    var_arr[i, :] = da.data
    var_det_arr[i, :] = detrend_anom_da.data
    var_STdet_arr[i,:] = detrend_stanom.data
        
    # print number b/c impatient. 
    if i%10==0: print(i, flush=True)
    
print('done with station datasets.', flush=True)    

/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/xarray/core/nputils.py:248: RankWarning: Polyfit may be poorly conditioned
  warn_on_deficient_rank(rank, x.shape[1])


0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530


/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/xarray/core/nputils.py:248: RankWarning: Polyfit may be poorly conditioned
  warn_on_deficient_rank(rank, x.shape[1])


540
550
560
570
580
590


/opt/sw/anaconda3/2023.09/envs/pangeo23/lib/python3.11/site-packages/xarray/core/nputils.py:248: RankWarning: Polyfit may be poorly conditioned
  warn_on_deficient_rank(rank, x.shape[1])


600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750
760
770
780
790
800
810
820
830
840
850
860
870
880
890
900
910
920
930
940
950
960
970
980
990
1000
1010
1020
1030
1040
1050
1060
1070
1080
1090
1100
1110
1120
1130
1140
1150
1160
1170
1180
1190
1200
1210
1220
1230
1240
1250
1260
1270
1280
1290
1300
1310
1320
1330
1340
1350
1360
1370
1380
1390
1400
1410
1420
1430
1440
1450
1460
1470
done with station datasets.
CPU times: user 44min 10s, sys: 880 ms, total: 44min 11s
Wall time: 3min 8s


In [14]:
varname = 'Tw'

In [16]:
# save prior attrs
attrs = ds_out.attrs

# put into DataArrays,
var_da = xr.DataArray(var_arr, dims=hw_ds.dims, coords=hw_ds.coords, name=varname)
var_det_da = xr.DataArray(var_det_arr, dims=hw_ds.dims, coords=hw_ds.coords, name=varname)
var_STdet_da = xr.DataArray(var_STdet_arr, dims=hw_ds.dims, coords=hw_ds.coords, name=varname)

# add attrs,
var_da.attrs = attrs
var_det_da.attrs = attrs
var_STdet_da.attrs = attrs

var_det_da.attrs['desc.'] = 'Data anomalized by removing the linear trend from each day of the year, linear trends generated using data smoothed with a centered 11-day rolling window'

var_STdet_da.attrs['desc.'] = 'Data anomalized by removing the linear trend from each day of the year, linear trends generated using data smoothed with a centered 11-day rolling window. Standard deviations generated for each day of the year, then smoothed with an 11-day rolling window as well. Stanom = detrend_anom / rolling_std.'

now = datetime.now()

for da2 in [var_da, var_det_da, var_STdet_da]:
    da2.attrs['script'] = script
    da2.attrs['timestamp'] = now.strftime("%Y-%m-%d %H:%M:%S")

#*#*#
# save
print('saving.', flush=True)
#var_da.to_dataset().to_netcdf('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.{}.nc'.format(varname))
var_det_da.to_dataset().to_netcdf('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.{}.detrend_anom.roll11.nc'.format(varname))
var_STdet_da.to_dataset().to_netcdf('/dx02/data/nsiegert/coastal_mhw_data/ALLSTATIONS.{}.detrend_stanom.roll11.nc'.format(varname))

print('dishes are done.', flush=True)

saving.
dishes are done.
